In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH = '/content/drive/MyDrive/project_factoria/waste_classifier'
print("Base path:", BASE_PATH)

Base path: /content/drive/MyDrive/project_factoria/waste_classifier


# Celda 2 — Instalar e importar librerías

In [ ]:
!pip install -q Pillow

import os
import shutil
import json
import csv
from pathlib import Path
from PIL import Image
import random
from collections import defaultdict, Counter

random.seed(42)
print("Librerías cargadas correctamente")

Librerías cargadas correctamente


# Celda 3 — Definir rutas de los 3 datasets y el mapeo a 7 clases


In [ ]:
# Rutas de los datasets raw
DS1 = os.path.join(BASE_PATH, 'datasets/raw/realwaste/RealWaste')
DS2 = os.path.join(BASE_PATH, 'datasets/raw/garbage_12cls/garbage_classification')
DS3 = os.path.join(BASE_PATH, 'datasets/raw/garbage_6cls/Garbage classification')

# Ruta destino del dataset unificado
UNIFIED = os.path.join(BASE_PATH, 'datasets/unified')
METADATA = os.path.join(BASE_PATH, 'datasets/metadata')

# Clases finales
CLASSES = ['carton', 'papel', 'metal', 'plastico', 'vidrio', 'organico', 'no_reciclable']

# Mapeo: (dataset_path, carpeta_original) -> clase_final
CLASS_MAP = {
    # --- DS1: RealWaste ---
    (DS1, 'Cardboard'):          'carton',
    (DS1, 'Paper'):              'papel',
    (DS1, 'Metal'):              'metal',
    (DS1, 'Plastic'):            'plastico',
    (DS1, 'Glass'):              'vidrio',
    (DS1, 'Food Organics'):      'organico',
    (DS1, 'Vegetation'):         'organico',
    (DS1, 'Miscellaneous Trash'):'no_reciclable',
    (DS1, 'Textile Trash'):      'no_reciclable',

    # --- DS2: garbage_12cls ---
    (DS2, 'cardboard'):          'carton',
    (DS2, 'paper'):              'papel',
    (DS2, 'metal'):              'metal',
    (DS2, 'plastic'):            'plastico',
    (DS2, 'brown-glass'):        'vidrio',
    (DS2, 'green-glass'):        'vidrio',
    (DS2, 'white-glass'):        'vidrio',
    (DS2, 'biological'):         'organico',
    (DS2, 'trash'):              'no_reciclable',
    (DS2, 'battery'):            'no_reciclable',
    (DS2, 'clothes'):            'no_reciclable',
    (DS2, 'shoes'):              'no_reciclable',

    # --- DS3: garbage_6cls ---
    (DS3, 'cardboard'):          'carton',
    (DS3, 'paper'):              'papel',
    (DS3, 'metal'):              'metal',
    (DS3, 'plastic'):            'plastico',
    (DS3, 'glass'):              'vidrio',
    (DS3, 'trash'):              'no_reciclable',
}

print("Mapeo definido:", len(CLASS_MAP), "carpetas → 7 clases")

Mapeo definido: 27 carpetas → 7 clases


# Celda 4 — Verificar que los 3 datasets existen y mostrar su contenido


In [ ]:
for nombre, ruta in [('RealWaste', DS1), ('garbage_12cls', DS2), ('garbage_6cls', DS3)]:
    if os.path.exists(ruta):
        carpetas = sorted(os.listdir(ruta))
        print(f"\n✓ {nombre} — {len(carpetas)} carpetas:")
        for c in carpetas:
            n = len(os.listdir(os.path.join(ruta, c)))
            print(f"   {c:<25} {n} archivos")
    else:
        print(f"\n✗ NO ENCONTRADO: {ruta}")
        print("  Revisa que la carpeta esté en la ruta correcta en Drive")


✓ RealWaste — 9 carpetas:
   Cardboard                 461 archivos
   Food Organics             411 archivos
   Glass                     420 archivos
   Metal                     790 archivos
   Miscellaneous Trash       495 archivos
   Paper                     500 archivos
   Plastic                   921 archivos
   Textile Trash             318 archivos
   Vegetation                436 archivos

✓ garbage_12cls — 12 carpetas:
   battery                   945 archivos
   biological                985 archivos
   brown-glass               607 archivos
   cardboard                 891 archivos
   clothes                   5325 archivos
   green-glass               629 archivos
   metal                     769 archivos
   paper                     1050 archivos
   plastic                   865 archivos
   shoes                     1977 archivos
   trash                     697 archivos
   white-glass               775 archivos

✓ garbage_6cls — 6 carpetas:
   cardboard              

# Celda 5 — Recopilar todas las imágenes y detectar corruptas


In [ ]:
EXTENSIONES_VALIDAS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

imagenes_validas   = []
imagenes_corruptas = []

for (ds_path, carpeta), clase in CLASS_MAP.items():
    carpeta_path = os.path.join(ds_path, carpeta)
    if not os.path.exists(carpeta_path):
        print(f"⚠ Carpeta no encontrada: {carpeta_path}")
        continue

    for archivo in os.listdir(carpeta_path):
        ext = Path(archivo).suffix.lower()
        if ext not in EXTENSIONES_VALIDAS:
            continue
        ruta_img = os.path.join(carpeta_path, archivo)
        # Solo verificamos tamaño 0 (archivos vacíos), sin abrir la imagen
        try:
            if os.path.getsize(ruta_img) > 0:
                imagenes_validas.append((ruta_img, clase))
            else:
                imagenes_corruptas.append(ruta_img)
        except Exception:
            imagenes_corruptas.append(ruta_img)

print(f"\nImágenes válidas:   {len(imagenes_validas)}")
print(f"Imágenes corruptas: {len(imagenes_corruptas)}")
if imagenes_corruptas:
    print("\nCorruptas:")
    for r in imagenes_corruptas:
        print(" ", r)


Imágenes válidas:   22794
Imágenes corruptas: 0


# Celda 6 — Ver distribución de clases antes del split




In [ ]:
conteo = Counter(clase for _, clase in imagenes_validas)

print("Distribución de clases (total):")
print(f"{'Clase':<18} {'Imágenes':>10}  {'%':>6}")
print("-" * 38)
total = sum(conteo.values())
for clase in CLASSES:
    n = conteo[clase]
    pct = n / total * 100
    barra = '█' * int(pct / 2)
    print(f"{clase:<18} {n:>10}   {pct:>5.1f}%  {barra}")
print(f"\n{'TOTAL':<18} {total:>10}")

Distribución de clases (total):
Clase                Imágenes       %
--------------------------------------
carton                   1755     7.7%  ███
papel                    2144     9.4%  ████
metal                    1969     8.6%  ████
plastico                 2268     9.9%  ████
vidrio                   2932    12.9%  ██████
organico                 1832     8.0%  ████
no_reciclable            9894    43.4%  █████████████████████

TOTAL                   22794


# Celda 7 — Se crea el split estratificado 80 / 10 / 10


In [ ]:
# Agrupar por clase
por_clase = defaultdict(list)
for ruta, clase in imagenes_validas:
    por_clase[clase].append(ruta)

# Barajar y partir cada clase por separado (split estratificado)
splits = {'train': [], 'val': [], 'test': []}

for clase in CLASSES:
    imgs = por_clase[clase].copy()
    random.shuffle(imgs)

    n = len(imgs)
    n_val  = max(1, int(n * 0.10))
    n_test = max(1, int(n * 0.10))
    n_train = n - n_val - n_test

    splits['train'] += [(r, clase) for r in imgs[:n_train]]
    splits['val']   += [(r, clase) for r in imgs[n_train:n_train+n_val]]
    splits['test']  += [(r, clase) for r in imgs[n_train+n_val:]]

print("Split estratificado:")
print(f"{'Split':<8} {'Total':>8}  Distribución por clase")
print("-" * 60)
for split, datos in splits.items():
    cnt = Counter(c for _, c in datos)
    resumen = '  '.join(f"{c[:3]}:{cnt[c]}" for c in CLASSES)
    print(f"{split:<8} {len(datos):>8}  {resumen}")

Split estratificado:
Split       Total  Distribución por clase
------------------------------------------------------------
train       18242  car:1405  pap:1716  met:1577  pla:1816  vid:2346  org:1466  no_:7916
val          2276  car:175  pap:214  met:196  pla:226  vid:293  org:183  no_:989
test         2276  car:175  pap:214  met:196  pla:226  vid:293  org:183  no_:989


# Celda 8 — Copia imágenes al dataset unificado

Esta celda tarda varios minutos porque copia ~22k imágenes a Drive. Ejecútala y déjala correr.

In [ ]:
import hashlib

def hash_nombre(ruta):
    """Genera nombre único basado en hash para evitar colisiones entre datasets."""
    h = hashlib.md5(ruta.encode()).hexdigest()[:10]
    ext = Path(ruta).suffix.lower()
    return f"{h}{ext}"

copiadas = 0
errores  = 0

for split, datos in splits.items():
    for ruta_origen, clase in datos:
        carpeta_dest = os.path.join(UNIFIED, split, clase)
        os.makedirs(carpeta_dest, exist_ok=True)

        nombre_dest = hash_nombre(ruta_origen)
        ruta_dest   = os.path.join(carpeta_dest, nombre_dest)

        if os.path.exists(ruta_dest):
            copiadas += 1
            continue

        try:
            shutil.copy2(ruta_origen, ruta_dest)
            copiadas += 1
        except Exception as e:
            errores += 1

        if copiadas % 1000 == 0:
            print(f"  Copiadas: {copiadas} / {total}...")

print(f"\n✓ Proceso terminado")
print(f"  Copiadas: {copiadas}")
print(f"  Errores:  {errores}")

  Copiadas: 1000 / 22794...
  Copiadas: 2000 / 22794...
  Copiadas: 3000 / 22794...
  Copiadas: 4000 / 22794...
  Copiadas: 5000 / 22794...
  Copiadas: 6000 / 22794...
  Copiadas: 7000 / 22794...
  Copiadas: 8000 / 22794...
  Copiadas: 9000 / 22794...
  Copiadas: 10000 / 22794...
  Copiadas: 11000 / 22794...
  Copiadas: 12000 / 22794...
  Copiadas: 13000 / 22794...
  Copiadas: 14000 / 22794...
  Copiadas: 15000 / 22794...
  Copiadas: 16000 / 22794...
  Copiadas: 17000 / 22794...
  Copiadas: 18000 / 22794...
  Copiadas: 19000 / 22794...
  Copiadas: 20000 / 22794...
  Copiadas: 21000 / 22794...
  Copiadas: 22000 / 22794...

✓ Proceso terminado
  Copiadas: 22794
  Errores:  0


# Celda 9 — Verificar estructura final del unified


In [ ]:
print("Estructura final en unified/:\n")
for split in ['train', 'val', 'test']:
    print(f"  {split}/")
    split_total = 0
    for clase in CLASSES:
        ruta = os.path.join(UNIFIED, split, clase)
        n = len(os.listdir(ruta)) if os.path.exists(ruta) else 0
        split_total += n
        print(f"    {clase:<18} {n:>6} imágenes")
    print(f"    {'TOTAL':<18} {split_total:>6}\n")

Estructura final en unified/:

  train/
    carton               1405 imágenes
    papel                1716 imágenes
    metal                1577 imágenes
    plastico             1816 imágenes
    vidrio               2346 imágenes
    organico             1466 imágenes
    no_reciclable        7916 imágenes
    TOTAL               18242

  val/
    carton                175 imágenes
    papel                 214 imágenes
    metal                 196 imágenes
    plastico              226 imágenes
    vidrio                293 imágenes
    organico              183 imágenes
    no_reciclable         989 imágenes
    TOTAL                2276

  test/
    carton                175 imágenes
    papel                 214 imágenes
    metal                 196 imágenes
    plastico              226 imágenes
    vidrio                293 imágenes
    organico              183 imágenes
    no_reciclable         989 imágenes
    TOTAL                2276



# Celda 10 — Guardar metadata CSV y class weights


In [ ]:
os.makedirs(METADATA, exist_ok=True)

# --- dataset.csv con todas las rutas y etiquetas ---
csv_path = os.path.join(METADATA, 'dataset.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ruta', 'clase', 'split'])
    for split, datos in splits.items():
        for ruta_origen, clase in datos:
            nombre_dest = hash_nombre(ruta_origen)
            ruta_unified = os.path.join(UNIFIED, split, clase, nombre_dest)
            writer.writerow([ruta_unified, clase, split])

print(f"✓ dataset.csv guardado en {csv_path}")

# --- class_weights.json (inverso de frecuencia, normalizado) ---
conteo_train = Counter(c for _, c in splits['train'])
total_train  = sum(conteo_train.values())
n_clases     = len(CLASSES)

weights = {}
for clase in CLASSES:
    weights[clase] = round(total_train / (n_clases * conteo_train[clase]), 4)

weights_path = os.path.join(METADATA, 'class_weights.json')
with open(weights_path, 'w') as f:
    json.dump(weights, f, indent=2)

print(f"✓ class_weights.json guardado")
print("\nClass weights:")
for clase, w in weights.items():
    print(f"  {clase:<18} {w:.4f}")

✓ dataset.csv guardado en /content/drive/MyDrive/project_factoria/waste_classifier/datasets/metadata/dataset.csv
✓ class_weights.json guardado

Class weights:
  carton             1.8548
  papel              1.5186
  metal              1.6525
  plastico           1.4350
  vidrio             1.1108
  organico           1.7776
  no_reciclable      0.3292
